# Package 2 — Predict customer lifetime (regression)

Target: `ltv_months`. Compare XGBoost, LightGBM, CatBoost with 5-fold cross-validation.

## Feature selection — and why

Excluded as leakage: `cumulative_profit`, `upsell`, `referred` — all three are outcomes
that unfold *during* the same relationship period as `ltv_months` itself, not known at
the moment a new customer is being evaluated. `purchased` is kept: it marks the start
of the relationship, not something that happens over its course.

Rows missing the target (`ltv_months`, 4 rows) are dropped — there is no valid stand-in
for an unknown answer.

In [1]:
import pandas as pd

df = pd.read_csv("../data/funnel_marketing_data.csv")
df = df.dropna(subset=["ltv_months"])

features = [
    "ad_budget", "num_leads", "leads_answered", "leads_not_answered",
    "followup_1", "followup_2", "followup_3", "followup_4", "followup_5",
    "not_closed", "closed", "calls_to_closed", "calls_to_not_closed",
    "customer_acquisition_cost", "purchased",
]

X = df[features]
y = df["ltv_months"]
X.shape, y.shape

((3496, 15), (3496,))

## Cross-validation setup

Same 5 folds are reused for every model, so the comparison between XGBoost,
LightGBM and CatBoost is fair (each sees the exact same train/test splits).
 with a fixed  so the split is randomized but reproducible.

In [2]:
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, r2_score
from xgboost import XGBRegressor
import numpy as np

kf = KFold(n_splits=5, shuffle=True, random_state=42)

rmse_scores = []
r2_scores = []

for train_idx, test_idx in kf.split(X):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    model = XGBRegressor(random_state=42)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    rmse_scores.append(np.sqrt(mean_squared_error(y_test, preds)))
    r2_scores.append(r2_score(y_test, preds))

print("RMSE per fold:", [round(s, 2) for s in rmse_scores])
print("R2 per fold:  ", [round(s, 2) for s in r2_scores])
print("Mean RMSE:", round(np.mean(rmse_scores), 2))
print("Mean R2:  ", round(np.mean(r2_scores), 2))

RMSE per fold: [np.float64(3.04), np.float64(3.48), np.float64(3.19), np.float64(3.32), np.float64(3.08)]
R2 per fold:   [0.94, 0.92, 0.93, 0.93, 0.94]
Mean RMSE: 3.22
Mean R2:   0.93


## Comparing XGBoost, LightGBM, CatBoost

Reusing the same  (same 5 folds, same ) so every model is
trained and tested on the exact same splits — a fair apples-to-apples comparison.

In [3]:
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

models = {
    "XGBoost": XGBRegressor(random_state=42),
    "LightGBM": LGBMRegressor(random_state=42, verbose=-1),
    "CatBoost": CatBoostRegressor(random_state=42, verbose=False),
}

results = {name: {"rmse": [], "r2": []} for name in models}

for train_idx, test_idx in kf.split(X):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    for name, model in models.items():
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        results[name]["rmse"].append(np.sqrt(mean_squared_error(y_test, preds)))
        results[name]["r2"].append(r2_score(y_test, preds))

summary = pd.DataFrame({
    name: {
        "mean_rmse": np.mean(scores["rmse"]),
        "mean_r2": np.mean(scores["r2"]),
    }
    for name, scores in results.items()
}).T

summary

,mean_rmse,mean_r2
XGBoost,3.223986,0.932658
LightGBM,2.991995,0.941860
CatBoost,2.945264,0.943669


## Feature importance comparison

CV was for an honest performance estimate; for interpretation we refit each model
once on the full dataset (more data -> more stable importance estimates).

Each library computes "importance" differently and on a different scale (LightGBM
default = split count, XGBoost/CatBoost = error-reduction based), so raw numbers are
not comparable across models. Each column is normalized to sum to 100%, so we compare
*relative ranking*, not raw units.

In [4]:
for model in models.values():
    model.fit(X, y)

importances = pd.DataFrame({
    name: model.feature_importances_
    for name, model in models.items()
}, index=X.columns)

importances_pct = importances.div(importances.sum(axis=0), axis=1) * 100
importances_pct.round(1).sort_values("CatBoost", ascending=False)

,XGBoost,LightGBM,CatBoost
calls_to_closed,59.9,7.1,52.6
ad_budget,0.2,3.6,21.8
num_leads,0.1,13.0,8.0
closed,37.9,2.6,4.1
followup_4,0.2,7.1,2.9
customer_acquisition_cost,0.3,7.0,2.8
followup_5,0.2,4.0,2.3
leads_not_answered,0.2,15.1,1.1
followup_2,0.1,6.1,1.0
leads_answered,0.1,10.4,0.8


## Sensitivity check: how much do `calls_to_closed` / `closed` drive the result?

Rerunning the exact same 5-fold CV, same 3 models, but dropping the two features
that dominated importance. If performance barely changes, the model was mostly
ignoring them despite the importance numbers; if it drops sharply, they really are
the main signal the model relies on.

In [5]:
features_no_close = [f for f in features if f not in ("calls_to_closed", "closed")]
X_v2 = df[features_no_close]

models_v2 = {
    "XGBoost": XGBRegressor(random_state=42),
    "LightGBM": LGBMRegressor(random_state=42, verbose=-1),
    "CatBoost": CatBoostRegressor(random_state=42, verbose=False),
}

results_v2 = {name: {"rmse": [], "r2": []} for name in models_v2}

for train_idx, test_idx in kf.split(X_v2):
    X_train, X_test = X_v2.iloc[train_idx], X_v2.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    for name, model in models_v2.items():
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        results_v2[name]["rmse"].append(np.sqrt(mean_squared_error(y_test, preds)))
        results_v2[name]["r2"].append(r2_score(y_test, preds))

summary_v2 = pd.DataFrame({
    name: {
        "mean_rmse": np.mean(scores["rmse"]),
        "mean_r2": np.mean(scores["r2"]),
    }
    for name, scores in results_v2.items()
}).T

comparison = summary.join(summary_v2, lsuffix="_with_closed", rsuffix="_without_closed")
comparison

,mean_rmse_with_closed,mean_r2_with_closed,mean_rmse_without_closed,mean_r2_without_closed
XGBoost,3.223986,0.932658,5.241219,0.822506
LightGBM,2.991995,0.941860,4.919728,0.843321
CatBoost,2.945264,0.943669,4.848883,0.847866


## Conclusions

**Chosen model: CatBoost.** Best mean RMSE (2.95 months) and R² (0.944) across 5-fold CV, though all three models (XGBoost, LightGBM, CatBoost) performed within a similar range — no single model was a dramatic outlier.

**Headline result:** the model predicts `ltv_months` within about 3 months on average, on a target that ranges 1–56 months (mean 22, std 12.5).

**Key finding — and an honest caveat:** `calls_to_closed` (how many calls it took to close the deal) and `closed` together account for the large majority of feature importance in XGBoost and CatBoost, and `calls_to_closed` alone correlates -0.65 with `ltv_months` — fewer calls to close strongly predicts a longer customer lifetime. This is not leakage in the strict sense (both describe the acquisition process, which completes before the LTV clock starts), but the concentration is unusually extreme for real-world data and likely reflects how this synthetic dataset was generated.

**Robustness check:** removing both features raises RMSE to ~4.85–5.24 months and drops R² to ~0.82–0.85. Performance degrades meaningfully but does not collapse — the remaining funnel/budget features (`ad_budget`, `num_leads`, the `followup_*` columns, etc.) still carry real, independent signal. Both versions are worth presenting: the full model as the practical best predictor, and the reduced model as evidence the result isn't a single-feature artifact.

**Business takeaway for Northbound Media:** how *hard* a customer was to close predicts their long-term value far more than how much was spent acquiring them or how many leads were generated — worth investigating *why* (e.g. does a fast, low-effort close signal a more decisive, higher-fit customer?).